# Project 2 — CORES for Monocular Depth Estimation

Kaggle-first implementation for studying convolutional-response OOD detection in a monocular depth estimation network.

**Planned domains:** NYU Depth v2 (ID) and KITTI (OOD).  
**Planned model:** FastDepth.  
**Main metrics:** AUROC, FPR95, RMSE, AbsRel, δ1, δ2, δ3.

> Start with `QUICK_MODE = True`. Dataset-specific code will be enabled after the exact Kaggle dataset sources and layouts have been selected.

## 1. Imports

In [ ]:
from __future__ import annotations

import json
import itertools
import copy
from functools import partial
import os
import platform
import random
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import roc_auc_score, roc_curve
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset, random_split

print = partial(print, flush=True)
from torchvision.models import MobileNet_V2_Weights, mobilenet_v2
from tqdm.auto import tqdm

print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')

## 2. Environment and configuration

In [ ]:
def detect_environment() -> str:
    if Path('/kaggle').exists():
        return 'kaggle'
    if 'google.colab' in sys.modules:
        return 'colab'
    return 'local'


ENVIRONMENT = detect_environment()

if ENVIRONMENT == 'kaggle':
    INPUT_ROOT = Path('/kaggle/input')
    WORK_ROOT = Path('/kaggle/working/cores-mde')
elif ENVIRONMENT == 'colab':
    INPUT_ROOT = Path('/content/data')
    WORK_ROOT = Path('/content/cores-mde')
else:
    INPUT_ROOT = Path.cwd().parent / 'data'
    WORK_ROOT = Path.cwd().parent / 'outputs'

WORK_ROOT.mkdir(parents=True, exist_ok=True)
(WORK_ROOT / 'checkpoints').mkdir(exist_ok=True)
(WORK_ROOT / 'figures').mkdir(exist_ok=True)
(WORK_ROOT / 'results').mkdir(exist_ok=True)

print(f'Environment: {ENVIRONMENT}')
print(f'Input root: {INPUT_ROOT}')
print(f'Work root: {WORK_ROOT}')

In [ ]:
@dataclass(frozen=True)
class Config:
    seed: int = 42
    quick_mode: bool = False
    train_model: bool = True
    load_checkpoint: bool = True
    image_height: int = 224
    image_width: int = 304
    batch_size: int = 8
    num_workers: int = 2
    epochs: int = 20
    learning_rate: float = 1e-4
    validation_fraction: float = 0.10
    early_stopping_patience: int = 5
    checkpoint_interval_batches: int = 250
    pretrained_encoder: bool = True
    nyu_dataset_dir: str = 'datasets/awsaf49/nyuv2-official-split-dataset'
    kitti_dataset_dir: str = 'datasets/artemmmtry/kitti-depth-prediction-evaluation'


CFG = Config()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def validate_cuda_compatibility() -> None:
    if not torch.cuda.is_available():
        return
    major, minor = torch.cuda.get_device_capability(0)
    device_arch = f'sm_{major}{minor}'
    supported_arches = set(torch.cuda.get_arch_list())
    if supported_arches and device_arch not in supported_arches:
        raise RuntimeError(
            f'GPU architecture {device_arch} is not supported by this PyTorch build. ' +
            'On Kaggle, switch from P100 to GPU T4 x2 and restart the session.'
        )


print(json.dumps(asdict(CFG), indent=2))
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA: {torch.version.cuda}')
    print(f'GPU count: {torch.cuda.device_count()}')
    validate_cuda_compatibility()
else:
    print('WARNING: GPU not detected. Enable a GPU accelerator for training.')

## 3. Reproducibility

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

    # Determinism improves reproducibility but may reduce performance.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(CFG.seed)
print(f'Global seed set to {CFG.seed}.')

## 4. Runtime diagnostics

In [ ]:
def runtime_report() -> dict[str, Any]:
    report: dict[str, Any] = {
        'environment': ENVIRONMENT,
        'platform': platform.platform(),
        'python': sys.version.split()[0],
        'torch': torch.__version__,
        'numpy': np.__version__,
        'sklearn': sklearn.__version__,
        'device': str(DEVICE),
        'cuda_available': torch.cuda.is_available(),
    }
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        report.update({
            'gpu': props.name,
            'gpu_memory_gib': round(props.total_memory / 1024**3, 2),
            'cuda': torch.version.cuda,
            'gpu_compute_capability': '.'.join(map(str, torch.cuda.get_device_capability(0))),
            'pytorch_cuda_arch_list': torch.cuda.get_arch_list(),
        })
    return report


RUNTIME = runtime_report()
print(json.dumps(RUNTIME, indent=2))
with (WORK_ROOT / 'runtime.json').open('w', encoding='utf-8') as file:
    json.dump(RUNTIME, file, indent=2)

## 5. Dataset discovery

This section deliberately fails early when dataset names have not been configured. It prevents a long Kaggle run from silently using the wrong folders.

In [ ]:
def list_input_datasets(root: Path) -> list[Path]:
    if not root.exists():
        return []
    return sorted(path for path in root.iterdir() if path.is_dir())


available_datasets = list_input_datasets(INPUT_ROOT)
print('Attached input datasets:')
for path in available_datasets:
    print(f'  - {path.name}')
if not available_datasets:
    print('  (none found — expected before Kaggle datasets are attached)')

In [ ]:
def resolve_dataset_root(input_root: Path, dataset_dir: str) -> Path:
    direct_path = input_root / dataset_dir
    if direct_path.is_dir():
        return direct_path

    # Kaggle may mount inputs as /kaggle/input/datasets/<owner>/<slug>.
    matches = sorted(
        path for path in input_root.rglob(dataset_dir) if path.is_dir()
    ) if input_root.exists() else []
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise RuntimeError(
            f'Multiple folders match {dataset_dir!r}: {[str(path) for path in matches]}'
        )
    return direct_path


NYU_ROOT = resolve_dataset_root(INPUT_ROOT, CFG.nyu_dataset_dir)
KITTI_ROOT = resolve_dataset_root(INPUT_ROOT, CFG.kitti_dataset_dir)

dataset_status = pd.DataFrame([
    {'dataset': 'NYU Depth v2', 'path': str(NYU_ROOT), 'found': NYU_ROOT.exists()},
    {'dataset': 'KITTI', 'path': str(KITTI_ROOT), 'found': KITTI_ROOT.exists()},
])
display(dataset_status)

DATASETS_READY = bool(dataset_status['found'].all())
if not DATASETS_READY:
    print('Dataset loaders remain disabled until the Config folder names are updated.')

In [ ]:
def compact_dataset_manifest(root: Path, max_examples: int = 30) -> pd.DataFrame:
    if not root.exists():
        return pd.DataFrame(columns=['relative_path', 'suffix', 'size_mib'])
    files = list(itertools.islice(
        (path for path in root.rglob('*') if path.is_file()), max_examples
    ))
    rows = [
        {
            'relative_path': str(path.relative_to(root)),
            'suffix': path.suffix.lower(),
            'size_mib': round(path.stat().st_size / 1024**2, 3),
        }
        for path in files
    ]
    print(f'{root.name}: showing up to {max_examples} discovered files')
    return pd.DataFrame(rows)


if NYU_ROOT.exists():
    display(compact_dataset_manifest(NYU_ROOT))
if KITTI_ROOT.exists():
    display(compact_dataset_manifest(KITTI_ROOT))

## 6. Core validation utilities

### NYU Depth v2 loader

RGB and depth files are paired by their shared identifier. Encoded uint16 depth values are mapped to the documented 0–10 metre interval. Depth remains at native resolution for evaluation.

In [ ]:
NYU_UINT16_MAX = float(2**16 - 1)
NYU_MAX_DEPTH_METERS = 10.0


def find_nyu_split_directory(root: Path, split: str) -> Path:
    if split == 'train':
        split_dir = root / 'train'
    elif split in {'test', 'val', 'validation'}:
        split_dir = root / 'test' / 'official'
    else:
        raise ValueError("split must be 'train' or 'test'.")
    if not split_dir.is_dir():
        raise FileNotFoundError(f'NYU split directory not found: {split_dir}')
    return split_dir


def discover_nyu_pairs(root: Path, split: str, limit=None):
    split_dir = find_nyu_split_directory(root, split)
    rgb_iterator = split_dir.rglob('rgb_*.png')
    if limit is None:
        rgb_paths = sorted(rgb_iterator)
    else:
        if limit <= 0:
            raise ValueError('limit must be positive when provided.')
        rgb_paths = list(itertools.islice(rgb_iterator, limit))
    pairs = [
        (rgb, rgb.with_name(rgb.name.replace('rgb_', 'depth_', 1)))
        for rgb in rgb_paths
    ]
    missing = [depth for _, depth in pairs if not depth.is_file()]
    if missing or not pairs:
        raise FileNotFoundError(
            f'Invalid NYU pairing: {len(pairs)} pairs, {len(missing)} missing depths.'
        )
    return pairs


class NYUDepthDataset(Dataset):
    def __init__(self, root: Path, split: str, output_size=(224, 304), limit=None):
        self.output_size = output_size
        self.pairs = discover_nyu_pairs(root, split, limit=limit)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        image_path, depth_path = self.pairs[index]
        with Image.open(image_path) as image_file:
            original_size = (image_file.height, image_file.width)
            image = rgb_to_tensor(image_file, self.output_size)
        with Image.open(depth_path) as depth_file:
            encoded_depth = np.asarray(depth_file, dtype=np.float32)
        depth = torch.from_numpy(
            encoded_depth / NYU_UINT16_MAX * NYU_MAX_DEPTH_METERS
        ).unsqueeze(0)
        return {
            'image': image,
            'depth': depth,
            'valid_mask': torch.isfinite(depth) & (depth > 0) & (depth <= 10.0),
            'original_size': original_size,
            'sample_id': image_path.stem.removeprefix('rgb_'),
        }

### KITTI validation loader

KITTI depth PNG values are converted to metres by dividing by 256. Zero-valued pixels remain invalid. RGB is resized for the network, while depth stays at native resolution so predictions can later be resized back for evaluation.

In [ ]:
KITTI_DEPTH_SCALE = 256.0


def kitti_groundtruth_name(image_name: str) -> str:
    marker = '_sync_image_'
    if marker not in image_name:
        raise ValueError(f'Unexpected KITTI validation RGB filename: {image_name}')
    return image_name.replace(marker, '_sync_groundtruth_depth_', 1)


def find_kitti_validation_directories(root: Path) -> tuple[Path, Path]:
    known_locations = (
        root / 'data_depth_selection' / 'depth_selection' / 'val_selection_cropped',
        root / 'depth_selection' / 'val_selection_cropped',
        root / 'val_selection_cropped',
    )
    selection_root = next((path for path in known_locations if path.is_dir()), None)
    if selection_root is None:
        candidates = sorted(
            path for path in root.rglob('val_selection_cropped') if path.is_dir()
        )
        if len(candidates) != 1:
            raise FileNotFoundError(
                f'Expected one val_selection_cropped below {root}, found {len(candidates)}.'
            )
        selection_root = candidates[0]
    image_dir = selection_root / 'image'
    depth_dir = selection_root / 'groundtruth_depth'
    if not image_dir.is_dir() or not depth_dir.is_dir():
        raise FileNotFoundError(f'Incomplete KITTI selection below {selection_root}.')
    return image_dir, depth_dir


def rgb_to_tensor(image: Image.Image, output_size: tuple[int, int]) -> torch.Tensor:
    array = np.asarray(image.convert('RGB'), dtype=np.float32) / 255.0
    tensor = torch.from_numpy(array).permute(2, 0, 1).unsqueeze(0)
    return F.interpolate(
        tensor, size=output_size, mode='bilinear', align_corners=False
    ).squeeze(0)


class KITTIDepthValidationDataset(Dataset):
    def __init__(self, root: Path, output_size=(224, 304), limit=None):
        self.output_size = output_size
        image_dir, depth_dir = find_kitti_validation_directories(root)
        images = sorted(image_dir.glob('*.png'))
        pairs = [
            (path, depth_dir / kitti_groundtruth_name(path.name))
            for path in images
        ]
        missing = [depth for _, depth in pairs if not depth.is_file()]
        if missing or not pairs:
            raise FileNotFoundError(
                f'Invalid KITTI pairing: {len(pairs)} pairs, {len(missing)} missing depths.'
            )
        self.pairs = pairs[:limit] if limit is not None else pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        image_path, depth_path = self.pairs[index]
        with Image.open(image_path) as image_file:
            original_size = (image_file.height, image_file.width)
            image = rgb_to_tensor(image_file, self.output_size)
        with Image.open(depth_path) as depth_file:
            depth = torch.from_numpy(
                np.asarray(depth_file, dtype=np.float32) / KITTI_DEPTH_SCALE
            ).unsqueeze(0)
        return {
            'image': image,
            'depth': depth,
            'valid_mask': depth > 0,
            'original_size': original_size,
            'sample_id': image_path.stem,
        }

In [ ]:
if KITTI_ROOT.exists():
    kitti_limit = 16 if CFG.quick_mode else None
    kitti_dataset = KITTIDepthValidationDataset(
        KITTI_ROOT,
        output_size=(CFG.image_height, CFG.image_width),
        limit=kitti_limit,
    )
    kitti_sample = kitti_dataset[0]
    validate_later = (kitti_sample['image'], kitti_sample['depth'])
    print(f'KITTI samples ready: {len(kitti_dataset)}')
    print(f"RGB input: {tuple(kitti_sample['image'].shape)}")
    print(f"Native depth: {tuple(kitti_sample['depth'].shape)}")
else:
    kitti_dataset = None
    print('Attach the KITTI Kaggle input to enable this loader.')

In [ ]:
if NYU_ROOT.exists():
    train_limit = 32 if CFG.quick_mode else None
    test_limit = 16 if CFG.quick_mode else None
    nyu_train_dataset = NYUDepthDataset(
        NYU_ROOT, 'train', (CFG.image_height, CFG.image_width), train_limit
    )
    nyu_test_dataset = NYUDepthDataset(
        NYU_ROOT, 'test', (CFG.image_height, CFG.image_width), test_limit
    )
    nyu_sample = nyu_test_dataset[0]
    print(f'NYU train samples ready: {len(nyu_train_dataset)}')
    print(f'NYU test samples ready: {len(nyu_test_dataset)}')
    print(f"RGB input: {tuple(nyu_sample['image'].shape)}")
    print(f"Native depth: {tuple(nyu_sample['depth'].shape)}")
    print(
        f"Valid depth range: {nyu_sample['depth'][nyu_sample['valid_mask']].min():.3f}–"
        f"{nyu_sample['depth'][nyu_sample['valid_mask']].max():.3f} m"
    )
else:
    nyu_train_dataset = nyu_test_dataset = None
    print('Attach the NYU Kaggle input to enable this loader.')

In [ ]:
if nyu_test_dataset is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(nyu_sample['image'].permute(1, 2, 0).numpy())
    axes[0].set_title('NYU RGB (network input)')
    depth_view = axes[1].imshow(nyu_sample['depth'].squeeze(0).numpy(), cmap='magma')
    axes[1].set_title('NYU depth (metres, native size)')
    fig.colorbar(depth_view, ax=axes[1], label='metres')
    for axis in axes:
        axis.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
def validate_rgb_depth_pair(image: torch.Tensor, depth: torch.Tensor) -> None:
    if image.ndim != 3 or image.shape[0] != 3:
        raise ValueError(f'Expected RGB tensor [3, H, W], got {tuple(image.shape)}')
    if depth.ndim not in (2, 3):
        raise ValueError(f'Expected depth tensor [H, W] or [1, H, W], got {tuple(depth.shape)}')
    depth_hw = depth.shape[-2:]
    if image.shape[-2:] != depth_hw:
        raise ValueError(f'RGB/depth spatial mismatch: {image.shape[-2:]} vs {depth_hw}')
    if not torch.isfinite(image).all():
        raise ValueError('RGB tensor contains NaN or infinity.')
    if not torch.isfinite(depth).all():
        raise ValueError('Depth tensor contains NaN or infinity.')


dummy_image = torch.rand(3, CFG.image_height, CFG.image_width)
dummy_depth = torch.rand(1, CFG.image_height, CFG.image_width)
validate_rgb_depth_pair(dummy_image, dummy_depth)
print('RGB/depth validation smoke test passed.')

## 7. Depth-estimation metrics

In [ ]:
DEPTH_METRIC_NAMES = ('rmse', 'abs_rel', 'delta1', 'delta2', 'delta3')


def _as_bhw(tensor: torch.Tensor, name: str) -> torch.Tensor:
    if tensor.ndim == 2:
        return tensor.unsqueeze(0)
    if tensor.ndim == 3:
        return tensor
    if tensor.ndim == 4 and tensor.shape[1] == 1:
        return tensor[:, 0]
    raise ValueError(
        f'{name} must have shape [H, W], [B, H, W], or [B, 1, H, W]; ' +
        f'got {tuple(tensor.shape)}'
    )


@torch.no_grad()
def compute_depth_metrics(
    prediction: torch.Tensor,
    target: torch.Tensor,
    *,
    min_depth: float = 1e-3,
    max_depth: float = 10.0,
) -> dict[str, float]:
    if min_depth <= 0 or max_depth <= min_depth:
        raise ValueError('Expected 0 < min_depth < max_depth.')

    prediction = _as_bhw(prediction.detach(), 'prediction').float()
    target = _as_bhw(target.detach(), 'target').float()
    if prediction.shape != target.shape:
        raise ValueError(
            f'Prediction/target shape mismatch: {tuple(prediction.shape)} vs ' +
            f'{tuple(target.shape)}'
        )

    per_image = {name: [] for name in DEPTH_METRIC_NAMES}
    for predicted_depth, target_depth in zip(prediction, target):
        valid = (
            torch.isfinite(predicted_depth)
            & torch.isfinite(target_depth)
            & (target_depth >= min_depth)
            & (target_depth <= max_depth)
        )
        if not torch.any(valid):
            continue

        pred = predicted_depth[valid].clamp(min=min_depth, max=max_depth)
        true = target_depth[valid]
        ratio = torch.maximum(true / pred, pred / true)
        per_image['rmse'].append(torch.sqrt(torch.mean((pred - true) ** 2)))
        per_image['abs_rel'].append(torch.mean(torch.abs(pred - true) / true))
        per_image['delta1'].append(torch.mean((ratio < 1.25).float()))
        per_image['delta2'].append(torch.mean((ratio < 1.25**2).float()))
        per_image['delta3'].append(torch.mean((ratio < 1.25**3).float()))

    if not per_image['rmse']:
        raise ValueError('The batch does not contain any valid depth pixels.')
    return {
        name: torch.stack(values).mean().item()
        for name, values in per_image.items()
    }

In [ ]:
# Deterministic smoke tests: these must pass before dataset evaluation.
perfect_target = torch.tensor([[[[1.0, 2.0], [4.0, 8.0]]]])
perfect_metrics = compute_depth_metrics(perfect_target.clone(), perfect_target)
assert perfect_metrics['rmse'] == 0.0
assert perfect_metrics['abs_rel'] == 0.0
assert perfect_metrics['delta1'] == 1.0
assert perfect_metrics['delta2'] == 1.0
assert perfect_metrics['delta3'] == 1.0

scaled_target = torch.ones(1, 1, 2, 2)
scaled_metrics = compute_depth_metrics(scaled_target * 2.0, scaled_target)
assert np.isclose(scaled_metrics['rmse'], 1.0)
assert np.isclose(scaled_metrics['abs_rel'], 1.0)
assert scaled_metrics['delta1'] == 0.0
assert scaled_metrics['delta2'] == 0.0
assert scaled_metrics['delta3'] == 0.0

display(pd.DataFrame([perfect_metrics, scaled_metrics], index=['perfect', '2x scale']))
print('Depth metric smoke tests passed.')

## 8. FastDepth baseline

This maintained adaptation uses a MobileNetV2 encoder and retains the central FastDepth design choices: a lightweight 5×5 depthwise-separable decoder, nearest-neighbour upsampling and additive skip connections. The original implementation used MobileNetV1 and PyTorch 0.4.1. Named intermediate features will later feed CORES.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def normalize_imagenet(images):
    if images.ndim != 4 or images.shape[1] != 3:
        raise ValueError(f'Expected [B, 3, H, W], got {tuple(images.shape)}')
    mean = images.new_tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
    std = images.new_tensor(IMAGENET_STD).view(1, 3, 1, 1)
    return (images - mean) / std

class DepthwiseDecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.depthwise = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 5, padding=2, groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels), nn.ReLU6(inplace=True),
        )
        self.pointwise = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU6(inplace=True),
        )

    def forward(self, inputs):
        return self.pointwise(self.depthwise(inputs))

class FastDepthMobileNetV2(nn.Module):
    def __init__(self, pretrained_encoder=True, max_depth=10.0):
        super().__init__()
        weights = MobileNet_V2_Weights.DEFAULT if pretrained_encoder else None
        self.encoder = mobilenet_v2(weights=weights).features
        self.decoder4 = DepthwiseDecoderBlock(1280, 96)
        self.decoder3 = DepthwiseDecoderBlock(96, 32)
        self.decoder2 = DepthwiseDecoderBlock(32, 24)
        self.decoder1 = DepthwiseDecoderBlock(24, 16)
        self.decoder0 = DepthwiseDecoderBlock(16, 16)
        self.depth_head = nn.Conv2d(16, 1, 1)
        self.max_depth = float(max_depth)

    @staticmethod
    def _upsample_add(inputs, skip):
        return F.interpolate(inputs, size=skip.shape[-2:], mode='nearest') + skip

    @staticmethod
    def _last_convolution(module):
        convolutions = [child for child in module.modules() if isinstance(child, nn.Conv2d)]
        if not convolutions:
            raise ValueError('Selected block has no convolution.')
        return convolutions[-1]

    def cores_response_modules(self):
        return {
            'encoder_early': self._last_convolution(self.encoder[3]),
            'encoder_middle': self._last_convolution(self.encoder[6]),
            'encoder_late': self._last_convolution(self.encoder[18]),
            'decoder_late': self.decoder0.pointwise[0],
        }

    def forward_features(self, inputs):
        skips, outputs = {}, inputs
        for index, layer in enumerate(self.encoder):
            outputs = layer(outputs)
            if index in {1, 3, 6, 13}:
                skips[index] = outputs
        late = outputs
        d4 = self._upsample_add(self.decoder4(late), skips[13])
        d3 = self._upsample_add(self.decoder3(d4), skips[6])
        d2 = self._upsample_add(self.decoder2(d3), skips[3])
        d1 = self._upsample_add(self.decoder1(d2), skips[1])
        d0 = self.decoder0(F.interpolate(d1, size=inputs.shape[-2:], mode='nearest'))
        return {'encoder_early': skips[3], 'encoder_middle': skips[6],
                'encoder_late': late, 'decoder_late': d0}

    def forward(self, inputs):
        features = self.forward_features(inputs)
        return torch.sigmoid(self.depth_head(features['decoder_late'])) * self.max_depth

In [ ]:
model = FastDepthMobileNetV2(
    pretrained_encoder=CFG.pretrained_encoder, max_depth=NYU_MAX_DEPTH_METERS
).to(DEVICE).eval()
parameter_count = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {parameter_count:,}')

with torch.inference_mode():
    sample_rgb = nyu_sample['image'].unsqueeze(0).to(DEVICE)
    normalized_rgb = normalize_imagenet(sample_rgb)
    sample_prediction = model(normalized_rgb)
    sample_features = model.forward_features(normalized_rgb)

assert sample_prediction.shape == (1, 1, CFG.image_height, CFG.image_width)
assert torch.isfinite(sample_prediction).all()
assert 0 <= sample_prediction.min() <= sample_prediction.max() <= 10
print(f'Prediction shape: {tuple(sample_prediction.shape)}')
print(f'Untrained depth range: {sample_prediction.min().item():.3f}–{sample_prediction.max().item():.3f} m')
for name, feature in sample_features.items():
    print(f'{name}: {tuple(feature.shape)}')
print('FastDepth forward smoke test passed.')

In [ ]:
prediction_native = F.interpolate(
    sample_prediction.cpu(), size=nyu_sample['depth'].shape[-2:],
    mode='bilinear', align_corners=False,
).squeeze()
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].imshow(nyu_sample['image'].permute(1, 2, 0).numpy())
axes[0].set_title('RGB')
axes[1].imshow(nyu_sample['depth'].squeeze().numpy(), cmap='magma', vmin=0, vmax=10)
axes[1].set_title('Ground truth')
axes[2].imshow(prediction_native.numpy(), cmap='magma', vmin=0, vmax=10)
axes[2].set_title('Untrained FastDepth output')
for axis in axes: axis.axis('off')
plt.tight_layout(); plt.show()

## 9. Training-pipeline smoke test

Before full training, the model must overfit one fixed mini-batch. This catches broken gradients, incorrect depth scaling and target-shape errors cheaply. The resulting weights are diagnostic only and are not used as the final model.

In [ ]:
def resize_depth_target(target, output_size, min_depth=1e-3, max_depth=10.0):
    valid = torch.isfinite(target) & (target >= min_depth) & (target <= max_depth)
    safe_target = torch.where(valid, target, torch.zeros_like(target))
    resized_target = F.interpolate(
        safe_target, size=output_size, mode='bilinear', align_corners=False
    )
    resized_valid = F.interpolate(
        valid.float(), size=output_size, mode='nearest'
    ).bool()
    return resized_target, resized_valid

def masked_l1_depth_loss(prediction, target, min_depth=1e-3, max_depth=10.0):
    resized_target, valid = resize_depth_target(
        target, prediction.shape[-2:], min_depth, max_depth
    )
    valid = valid & torch.isfinite(prediction)
    if not torch.any(valid):
        raise ValueError('No valid depth pixels in this batch.')
    return torch.abs(prediction[valid] - resized_target[valid]).mean()

perfect = torch.tensor([[[[1.0, 2.0], [3.0, 4.0]]]])
assert masked_l1_depth_loss(perfect, perfect).item() == 0.0
print('Masked depth loss smoke test passed.')

In [ ]:
smoke_loader = DataLoader(
    nyu_train_dataset, batch_size=2, shuffle=False, num_workers=CFG.num_workers,
    pin_memory=(DEVICE.type == 'cuda'),
)
smoke_batch = next(iter(smoke_loader))
smoke_images = smoke_batch['image'].to(DEVICE, non_blocking=True)
smoke_targets = smoke_batch['depth'].to(DEVICE, non_blocking=True)
overfit_model = copy.deepcopy(model).train()
optimizer = torch.optim.AdamW(overfit_model.parameters(), lr=1e-3, weight_decay=0.0)
smoke_losses = []

for step in range(25):
    optimizer.zero_grad(set_to_none=True)
    predictions = overfit_model(normalize_imagenet(smoke_images))
    loss = masked_l1_depth_loss(predictions, smoke_targets)
    loss.backward()
    optimizer.step()
    smoke_losses.append(loss.item())

print(f'Initial single-batch loss: {smoke_losses[0]:.4f} m')
print(f'Final single-batch loss: {smoke_losses[-1]:.4f} m')
assert np.isfinite(smoke_losses).all()
assert smoke_losses[-1] < smoke_losses[0], 'Single-batch loss did not decrease.'
plt.figure(figsize=(7, 3))
plt.plot(smoke_losses, marker='o', markersize=3)
plt.xlabel('Optimization step'); plt.ylabel('Masked L1 (m)')
plt.title('Single-batch overfit test'); plt.grid(alpha=0.3); plt.show()
print('Single-batch overfit test passed.')
del overfit_model, optimizer, smoke_images, smoke_targets
torch.cuda.empty_cache()

## 10. Baseline training and validation

A deterministic validation subset is carved out of the NYU training split. The official test split remains untouched until final evaluation. Full mode uses all available training pairs for 20 epochs, with early stopping and resumable checkpoints.

In [ ]:
validation_size = max(1, round(len(nyu_train_dataset) * CFG.validation_fraction))
training_size = len(nyu_train_dataset) - validation_size
split_generator = torch.Generator().manual_seed(CFG.seed)
training_dataset, validation_dataset = random_split(
    nyu_train_dataset, [training_size, validation_size], generator=split_generator
)
train_loader = DataLoader(
    training_dataset, batch_size=CFG.batch_size, shuffle=True,
    num_workers=CFG.num_workers, pin_memory=(DEVICE.type == 'cuda'),
    persistent_workers=(CFG.num_workers > 0),
)
validation_loader = DataLoader(
    validation_dataset, batch_size=CFG.batch_size, shuffle=False,
    num_workers=CFG.num_workers, pin_memory=(DEVICE.type == 'cuda'),
    persistent_workers=(CFG.num_workers > 0),
)
test_loader = DataLoader(
    nyu_test_dataset, batch_size=CFG.batch_size, shuffle=False,
    num_workers=CFG.num_workers, pin_memory=(DEVICE.type == 'cuda'),
    persistent_workers=(CFG.num_workers > 0),
)
print(f'Train/validation/test samples: {training_size}/{validation_size}/{len(nyu_test_dataset)}')
print(f'Train/validation/test batches: {len(train_loader)}/{len(validation_loader)}/{len(test_loader)}')

In [ ]:
def train_one_epoch(model, loader, optimizer, epoch, checkpoint_callback=None):
    model.train()
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))
    total_loss = 0.0
    total_samples = 0
    for batch_index, batch in enumerate(loader, start=1):
        images = batch['image'].to(DEVICE, non_blocking=True)
        targets = batch['depth'].to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == 'cuda')):
            predictions = model(normalize_imagenet(images))
            loss = masked_l1_depth_loss(predictions, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * images.shape[0]
        total_samples += images.shape[0]
        if (
            checkpoint_callback is not None
            and batch_index % CFG.checkpoint_interval_batches == 0
        ):
            running_loss = total_loss / total_samples
            checkpoint_callback(epoch, batch_index, running_loss)
            print(
                f'  Epoch {epoch:02d}: batch {batch_index}/{len(loader)}, '
                f'running loss={running_loss:.4f} m'
            )
    return total_loss / total_samples

@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    totals = {name: 0.0 for name in ('rmse', 'abs_rel', 'delta1', 'delta2', 'delta3')}
    total_loss = 0.0
    total_samples = 0
    for batch in loader:
        images = batch['image'].to(DEVICE, non_blocking=True)
        targets = batch['depth'].to(DEVICE, non_blocking=True)
        predictions = model(normalize_imagenet(images))
        loss = masked_l1_depth_loss(predictions, targets)
        resized_targets, _ = resize_depth_target(targets, predictions.shape[-2:])
        metrics = compute_depth_metrics(predictions, resized_targets)
        batch_size = images.shape[0]
        total_loss += loss.item() * batch_size
        total_samples += batch_size
        for name, value in metrics.items():
            totals[name] += value * batch_size
    return total_loss / total_samples, {name: value / total_samples for name, value in totals.items()}

In [ ]:
checkpoint_path = WORK_ROOT / 'checkpoints' / 'best_fastdepth_nyu.pt'
last_checkpoint_path = WORK_ROOT / 'checkpoints' / 'last_fastdepth_nyu.pt'
emergency_checkpoint_path = WORK_ROOT / 'checkpoints' / 'emergency_fastdepth_nyu.pt'
optimizer = torch.optim.AdamW(
    model.parameters(), lr=CFG.learning_rate, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)
history = []
best_validation_loss = float('inf')
start_epoch = 1

def checkpoint_payload(epoch):
    return {
        'epoch': epoch, 'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_validation_loss': best_validation_loss, 'history': history,
        'config': asdict(CFG),
    }

def save_mid_epoch_checkpoint(epoch, batch_index, running_loss):
    payload = checkpoint_payload(epoch)
    payload.update({
        'batch_index': batch_index, 'running_train_loss': running_loss,
        'checkpoint_kind': 'mid_epoch',
    })
    torch.save(payload, last_checkpoint_path)

resume_candidates = [last_checkpoint_path]
resume_candidates += [
    INPUT_ROOT / 'datasets/plomo02/project2-cv-checkpoints/last_fastdepth_nyu.pt'
]
resume_candidates += list(INPUT_ROOT.glob('*/last_fastdepth_nyu.pt'))
resume_candidates += list(INPUT_ROOT.glob('*/cores-mde/checkpoints/last_fastdepth_nyu.pt'))
resume_path = next((path for path in resume_candidates if path.is_file()), None)
if CFG.load_checkpoint and resume_path is not None:
    resume_state = torch.load(resume_path, map_location=DEVICE, weights_only=False)
    saved_config = resume_state.get('config', {})
    compatible = (
        saved_config.get('quick_mode') == CFG.quick_mode
        and saved_config.get('image_height') == CFG.image_height
        and saved_config.get('image_width') == CFG.image_width
    )
    if compatible:
        model.load_state_dict(resume_state['model_state_dict'])
        optimizer.load_state_dict(resume_state['optimizer_state_dict'])
        if 'scheduler_state_dict' in resume_state:
            scheduler.load_state_dict(resume_state['scheduler_state_dict'])
        history = resume_state.get('history', [])
        best_validation_loss = resume_state.get('best_validation_loss', float('inf'))
        saved_epoch = int(resume_state['epoch'])
        start_epoch = (
            saved_epoch if resume_state.get('checkpoint_kind') == 'mid_epoch'
            else saved_epoch + 1
        )
        print(f'Resuming from {resume_path}, epoch {start_epoch}.')
    else:
        print(f'Ignoring incompatible checkpoint: {resume_path}')

current_epoch = start_epoch - 1
epochs_without_improvement = 0
if CFG.train_model:
    try:
        for epoch in range(start_epoch, CFG.epochs + 1):
            current_epoch = epoch
            train_loss = train_one_epoch(
                model, train_loader, optimizer, epoch, save_mid_epoch_checkpoint
            )
            validation_loss, validation_metrics = evaluate_model(model, validation_loader)
            record = {'epoch': epoch, 'train_loss': train_loss,
                      'validation_loss': validation_loss, **validation_metrics}
            history.append(record)
            scheduler.step(validation_loss)
            print(
                f'Epoch {epoch:02d}/{CFG.epochs}: train={train_loss:.4f} m, '
                f'val={validation_loss:.4f} m, RMSE={validation_metrics["rmse"]:.4f}, '
                f'AbsRel={validation_metrics["abs_rel"]:.4f}, '
                f'delta1={validation_metrics["delta1"]:.4f}'
            )
            if validation_loss < best_validation_loss:
                best_validation_loss = validation_loss
                epochs_without_improvement = 0
                torch.save(checkpoint_payload(epoch), checkpoint_path)
                print(f'  Saved best checkpoint: {checkpoint_path}')
            else:
                epochs_without_improvement += 1
            torch.save(checkpoint_payload(epoch), last_checkpoint_path)
            print(f'  Saved recovery checkpoint: {last_checkpoint_path}')
            if epochs_without_improvement >= CFG.early_stopping_patience:
                print(f'Early stopping after {epoch} epochs.')
                break
    except BaseException:
        torch.save(checkpoint_payload(current_epoch), emergency_checkpoint_path)
        print(f'Emergency checkpoint saved: {emergency_checkpoint_path}')
        raise
elif CFG.load_checkpoint and checkpoint_path.is_file():
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    history = checkpoint.get('history', [])
    best_validation_loss = checkpoint['best_validation_loss']
else:
    print('Training disabled and no checkpoint found; evaluating initial weights.')

best_candidates = [checkpoint_path]
best_candidates += [
    INPUT_ROOT / 'datasets/plomo02/project2-cv-checkpoints/best_fastdepth_nyu.pt'
]
best_candidates += list(INPUT_ROOT.glob('*/best_fastdepth_nyu.pt'))
best_candidates += list(INPUT_ROOT.glob('*/cores-mde/checkpoints/best_fastdepth_nyu.pt'))
best_path = next((path for path in best_candidates if path.is_file()), None)
if best_path is not None:
    best_checkpoint = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(best_checkpoint['model_state_dict'])
final_validation_loss, final_validation_metrics = evaluate_model(model, validation_loader)
test_loss, test_metrics = evaluate_model(model, test_loader)
validation_results = {'l1': final_validation_loss, **final_validation_metrics}
test_results = {'l1': test_loss, **test_metrics}
results_frame = pd.DataFrame(
    [validation_results, test_results], index=['NYU validation', 'NYU official test']
)
display(results_frame.round(4))
results_directory = WORK_ROOT / 'results'
results_directory.mkdir(parents=True, exist_ok=True)
results_frame.to_csv(results_directory / 'fastdepth_depth_metrics.csv')
if history:
    pd.DataFrame(history).to_csv(results_directory / 'fastdepth_training_history.csv', index=False)
with (results_directory / 'fastdepth_depth_metrics.json').open('w') as file:
    json.dump(results_frame.to_dict(orient='index'), file, indent=2)
assert np.isfinite(results_frame.to_numpy()).all()
print('Baseline training/evaluation pipeline passed.')

In [ ]:
if history:
    history_frame = pd.DataFrame(history)
    display(history_frame.round(4))
    history_frame.plot(x='epoch', y=['train_loss', 'validation_loss'], marker='o')
    plt.ylabel('Masked L1 (m)'); plt.title('FastDepth learning curves')
    plt.grid(alpha=0.3); plt.show()

visual_batch = next(iter(validation_loader))
visual_images = visual_batch['image'][:3].to(DEVICE)
with torch.no_grad():
    visual_predictions = model(normalize_imagenet(visual_images)).cpu()
visual_targets = visual_batch['depth'][:3]
fig, axes = plt.subplots(len(visual_images), 3, figsize=(12, 4 * len(visual_images)))
for row in range(len(visual_images)):
    axes[row, 0].imshow(visual_images[row].cpu().permute(1, 2, 0).clamp(0, 1))
    axes[row, 1].imshow(visual_targets[row, 0], cmap='magma', vmin=0, vmax=10)
    axes[row, 2].imshow(visual_predictions[row, 0], cmap='magma', vmin=0, vmax=10)
    for column, title in enumerate(('RGB', 'Ground truth', 'Prediction')):
        axes[row, column].set_title(title); axes[row, column].axis('off')
plt.tight_layout(); plt.show()

## 11. CORES adaptation for dense depth estimation

The original CORES method targets classifiers. It measures the magnitude and frequency of extreme positive and negative **pre-activation convolution responses**, with higher scores interpreted as more ID-like. FastDepth has no class logits, so the classifier-specific backtracking step is excluded from this first MDE baseline and all kernels are scored. This adaptation is reported explicitly and sample-relevant kernel selection remains a later ablation. Thresholds and layer normalization are calibrated only on NYU validation; NYU official test and KITTI remain evaluation-only.

In [ ]:
def single_layer_cores(responses, tau_positive, tau_negative,
                       lambda_magnitude=10.0, lambda_frequency=1.0, epsilon=1e-12):
    if responses.ndim != 4:
        raise ValueError(f'Expected [B,C,H,W], got {tuple(responses.shape)}')
    channel_maxima = responses.amax(dim=(-2, -1))
    channel_minima = responses.amin(dim=(-2, -1))
    rm_positive = F.relu(channel_maxima - tau_positive).mean(dim=1)
    rm_negative = F.relu(tau_negative - channel_minima).mean(dim=1)
    rf_positive = (channel_maxima > tau_positive).float().mean(dim=1)
    rf_negative = (channel_minima < tau_negative).float().mean(dim=1)
    log_score = lambda_magnitude * (
        torch.log(rm_positive.clamp_min(epsilon))
        + torch.log(rm_negative.clamp_min(epsilon))
    ) + lambda_frequency * (
        torch.log(rf_positive.clamp_min(epsilon))
        + torch.log(rf_negative.clamp_min(epsilon))
    )
    return {'rm_positive': rm_positive, 'rm_negative': rm_negative,
            'rf_positive': rf_positive, 'rf_negative': rf_negative,
            'log_score': log_score}

controlled = torch.tensor([[[[2., 0.], [-2., 0.]], [[3., 1.], [-4., 0.]]]])
controlled_score = single_layer_cores(controlled, 1.0, -1.0, 1.0, 1.0)
assert np.isclose(controlled_score['log_score'].exp().item(), 3.0)
print('Single-layer CORES synthetic test passed.')

In [ ]:
response_modules = model.cores_response_modules()

@torch.no_grad()
def capture_convolution_responses(images):
    captured, handles = {}, []
    def make_hook(name):
        def hook(_module, _inputs, output):
            captured[name] = output.detach()
        return hook
    try:
        for name, module in response_modules.items():
            handles.append(module.register_forward_hook(make_hook(name)))
        model(normalize_imagenet(images))
    finally:
        for handle in handles:
            handle.remove()
    missing = set(response_modules) - set(captured)
    if missing:
        raise RuntimeError(f'Missing CORES responses: {sorted(missing)}')
    return captured

probe_responses = capture_convolution_responses(visual_images[:1])
for name, response in probe_responses.items():
    print(f'{name}: {tuple(response.shape)}, range={response.min():.3f}..{response.max():.3f}')
    assert response.min() < 0 < response.max(), f'{name} lacks signed pre-activation responses.'
print('Signed convolution-response capture passed.')

In [ ]:
@torch.no_grad()
def calibrate_cores_thresholds(loader, max_batches=32):
    maxima = {name: [] for name in response_modules}
    minima = {name: [] for name in response_modules}
    for batch_index, batch in enumerate(loader):
        if batch_index >= max_batches:
            break
        images = batch['image'].to(DEVICE, non_blocking=True)
        for name, response in capture_convolution_responses(images).items():
            maxima[name].append(response.amax(dim=(-2, -1)).cpu().flatten())
            minima[name].append(response.amin(dim=(-2, -1)).cpu().flatten())
    thresholds = {}
    for name in response_modules:
        thresholds[name] = {
            'tau_positive': torch.cat(maxima[name]).quantile(0.50).item(),
            'tau_negative': torch.cat(minima[name]).quantile(0.50).item(),
        }
    return thresholds

cores_thresholds = calibrate_cores_thresholds(validation_loader)
display(pd.DataFrame(cores_thresholds).T.round(4))
with (results_directory / 'cores_thresholds.json').open('w') as file:
    json.dump(cores_thresholds, file, indent=2)
print('CORES thresholds calibrated on NYU validation only.')

In [ ]:
@torch.no_grad()
def collect_cores_scores(loader, max_batches=None):
    scores = {name: [] for name in response_modules}
    for batch_index, batch in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break
        images = batch['image'].to(DEVICE, non_blocking=True)
        responses = capture_convolution_responses(images)
        for name, response in responses.items():
            threshold = cores_thresholds[name]
            components = single_layer_cores(
                response, threshold['tau_positive'], threshold['tau_negative']
            )
            scores[name].append(components['log_score'].cpu())
    return {name: torch.cat(values).numpy() for name, values in scores.items()}

kitti_ood_loader = DataLoader(
    kitti_dataset, batch_size=CFG.batch_size, shuffle=False,
    num_workers=CFG.num_workers, pin_memory=(DEVICE.type == 'cuda'),
    persistent_workers=(CFG.num_workers > 0),
)
calibration_scores = collect_cores_scores(validation_loader, max_batches=32)
id_scores = collect_cores_scores(test_loader)
ood_scores = collect_cores_scores(kitti_ood_loader)

def ood_metrics(high_id_scores, low_id_scores):
    labels = np.concatenate([np.ones(len(high_id_scores)), np.zeros(len(low_id_scores))])
    scores = np.concatenate([high_id_scores, low_id_scores])
    fpr, tpr, _ = roc_curve(labels, scores, pos_label=1)
    index = np.flatnonzero(tpr >= 0.95)[0]
    return {'auroc': roc_auc_score(labels, scores), 'fpr95': fpr[index]}

normalized_id, normalized_ood, metric_rows = {}, {}, []
for name in response_modules:
    center = calibration_scores[name].mean()
    scale = calibration_scores[name].std() + 1e-12
    normalized_id[name] = (id_scores[name] - center) / scale
    normalized_ood[name] = (ood_scores[name] - center) / scale
    metric_rows.append({'configuration': name, **ood_metrics(id_scores[name], ood_scores[name])})
multi_id = np.mean(np.stack(list(normalized_id.values())), axis=0)
multi_ood = np.mean(np.stack(list(normalized_ood.values())), axis=0)
metric_rows.append({'configuration': 'multi_layer_mean', **ood_metrics(multi_id, multi_ood)})
cores_metrics_frame = pd.DataFrame(metric_rows).set_index('configuration')
display((cores_metrics_frame * 100).round(2))
cores_metrics_frame.to_csv(results_directory / 'cores_ood_metrics.csv')

score_rows = []
for domain, domain_scores in [('NYU_test_ID', id_scores), ('KITTI_OOD', ood_scores)]:
    for layer, values in domain_scores.items():
        score_rows.extend({'domain': domain, 'layer': layer, 'score': value} for value in values)
pd.DataFrame(score_rows).to_csv(results_directory / 'cores_scores.csv', index=False)
assert np.isfinite(cores_metrics_frame.to_numpy()).all()
print('CORES NYU/KITTI evaluation passed.')

In [ ]:
def response_selected_cores(responses, tau_positive, tau_negative, fraction=0.20):
    channel_count = responses.shape[1]
    selected_count = max(1, round(channel_count * fraction))
    maxima = responses.amax(dim=(-2, -1))
    minima = responses.amin(dim=(-2, -1))
    positive_indices = maxima.topk(selected_count, dim=1).indices
    negative_indices = minima.topk(selected_count, dim=1, largest=False).indices
    spatial_shape = responses.shape[-2:]
    positive = responses.gather(
        1, positive_indices[..., None, None].expand(-1, -1, *spatial_shape)
    )
    negative = responses.gather(
        1, negative_indices[..., None, None].expand(-1, -1, *spatial_shape)
    )
    positive_parts = single_layer_cores(positive, tau_positive, tau_negative)
    negative_parts = single_layer_cores(negative, tau_positive, tau_negative)
    eps = 1e-12
    return 10 * (
        torch.log(positive_parts['rm_positive'].clamp_min(eps))
        + torch.log(negative_parts['rm_negative'].clamp_min(eps))
    ) + (
        torch.log(positive_parts['rf_positive'].clamp_min(eps))
        + torch.log(negative_parts['rf_negative'].clamp_min(eps))
    )

@torch.no_grad()
def collect_cores_ablation(loader):
    collected = {}
    eps = 1e-12
    for batch in loader:
        images = batch['image'].to(DEVICE, non_blocking=True)
        for layer, response in capture_convolution_responses(images).items():
            threshold = cores_thresholds[layer]
            parts = single_layer_cores(
                response, threshold['tau_positive'], threshold['tau_negative']
            )
            logs = {name: torch.log(parts[name].clamp_min(eps))
                    for name in ('rm_positive', 'rm_negative', 'rf_positive', 'rf_negative')}
            variants = {
                'all': parts['log_score'],
                'magnitude_only': 10 * (logs['rm_positive'] + logs['rm_negative']),
                'frequency_only': logs['rf_positive'] + logs['rf_negative'],
                'positive_only': 10 * logs['rm_positive'] + logs['rf_positive'],
                'negative_only': 10 * logs['rm_negative'] + logs['rf_negative'],
                'response_selected_20pct': response_selected_cores(
                    response, threshold['tau_positive'], threshold['tau_negative']
                ),
            }
            for variant, values in variants.items():
                collected.setdefault((layer, variant), []).append(values.cpu())
    return {key: torch.cat(values).numpy() for key, values in collected.items()}

id_ablation_scores = collect_cores_ablation(test_loader)
ood_ablation_scores = collect_cores_ablation(kitti_ood_loader)
ablation_rows = []
for (layer, variant), values in id_ablation_scores.items():
    ablation_rows.append({
        'layer': layer, 'variant': variant,
        **ood_metrics(values, ood_ablation_scores[(layer, variant)]),
    })
cores_ablation_frame = pd.DataFrame(ablation_rows).sort_values(
    ['auroc', 'fpr95'], ascending=[False, True]
)
display(cores_ablation_frame.assign(
    auroc=cores_ablation_frame.auroc * 100,
    fpr95=cores_ablation_frame.fpr95 * 100,
).round(2))
cores_ablation_frame.to_csv(results_directory / 'cores_component_ablation.csv', index=False)
print('CORES component and response-selection ablation passed.')

In [ ]:
@torch.no_grad()
def collect_rgb_statistics(loader, max_batches=None):
    features = []
    for batch_index, batch in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break
        images = batch['image'].float()
        channel_mean = images.mean(dim=(-2, -1))
        channel_std = images.std(dim=(-2, -1))
        brightness = images.mean(dim=(1, 2, 3), keepdim=False).unsqueeze(1)
        saturation = (images.max(dim=1).values - images.min(dim=1).values).mean(
            dim=(-2, -1), keepdim=False
        ).unsqueeze(1)
        features.append(torch.cat([channel_mean, channel_std, brightness, saturation], dim=1))
    return torch.cat(features).numpy()

rgb_calibration = collect_rgb_statistics(validation_loader, max_batches=32)
rgb_id = collect_rgb_statistics(test_loader)
rgb_ood = collect_rgb_statistics(kitti_ood_loader)
rgb_center = rgb_calibration.mean(axis=0)
rgb_covariance = np.cov(rgb_calibration, rowvar=False)
rgb_precision = np.linalg.pinv(rgb_covariance + np.eye(rgb_covariance.shape[0]) * 1e-5)
def negative_mahalanobis(values):
    centered = values - rgb_center
    return -np.einsum('bi,ij,bj->b', centered, rgb_precision, centered)
rgb_id_scores = negative_mahalanobis(rgb_id)
rgb_ood_scores = negative_mahalanobis(rgb_ood)
print('RGB-statistics baseline:', ood_metrics(rgb_id_scores, rgb_ood_scores))

def bootstrap_metrics(id_values, ood_values, repetitions=500, seed=CFG.seed):
    generator = np.random.default_rng(seed)
    samples = {'auroc': [], 'fpr95': []}
    point = ood_metrics(id_values, ood_values)
    for _ in range(repetitions):
        sampled_id = id_values[generator.integers(0, len(id_values), len(id_values))]
        sampled_ood = ood_values[generator.integers(0, len(ood_values), len(ood_values))]
        metrics = ood_metrics(sampled_id, sampled_ood)
        for name in samples:
            samples[name].append(metrics[name])
    row = {}
    for name, values in samples.items():
        row[name] = point[name]
        row[f'{name}_low'] = np.quantile(values, 0.025)
        row[f'{name}_high'] = np.quantile(values, 0.975)
    return row

robustness_scores = {
    'CORES middle magnitude': (
        id_ablation_scores[('encoder_middle', 'magnitude_only')],
        ood_ablation_scores[('encoder_middle', 'magnitude_only')],
    ),
    'CORES middle full': (id_scores['encoder_middle'], ood_scores['encoder_middle']),
    'CORES multi-layer': (multi_id, multi_ood),
    'RGB Mahalanobis': (rgb_id_scores, rgb_ood_scores),
}
robustness_rows = [
    {'configuration': name, **bootstrap_metrics(id_values, ood_values)}
    for name, (id_values, ood_values) in robustness_scores.items()
]
cores_robustness_frame = pd.DataFrame(robustness_rows).set_index('configuration')
display((cores_robustness_frame * 100).round(2))
cores_robustness_frame.to_csv(results_directory / 'cores_bootstrap_rgb_baseline.csv')
assert np.isfinite(cores_robustness_frame.to_numpy()).all()
print('Bootstrap confidence intervals and RGB baseline passed.')

In [ ]:
@torch.no_grad()
def capture_middle_response(target_model, images):
    captured = {}
    module = target_model.cores_response_modules()['encoder_middle']
    handle = module.register_forward_hook(
        lambda _module, _inputs, output: captured.setdefault('middle', output.detach())
    )
    try:
        target_model(normalize_imagenet(images))
    finally:
        handle.remove()
    return captured['middle']

@torch.no_grad()
def calibrate_middle_magnitude(target_model, loader, max_batches=32):
    maxima, minima = [], []
    for batch_index, batch in enumerate(loader):
        if batch_index >= max_batches:
            break
        response = capture_middle_response(
            target_model, batch['image'].to(DEVICE, non_blocking=True)
        )
        maxima.append(response.amax(dim=(-2, -1)).cpu().flatten())
        minima.append(response.amin(dim=(-2, -1)).cpu().flatten())
    return torch.cat(maxima).quantile(0.5).item(), torch.cat(minima).quantile(0.5).item()

@torch.no_grad()
def collect_middle_magnitude(target_model, loader, thresholds, transform=None):
    values = []
    tau_positive, tau_negative = thresholds
    for batch in loader:
        images = batch['image'].to(DEVICE, non_blocking=True)
        if transform is not None:
            images = transform(images)
        response = capture_middle_response(target_model, images)
        parts = single_layer_cores(response, tau_positive, tau_negative)
        values.append((10 * (
            torch.log(parts['rm_positive'].clamp_min(1e-12))
            + torch.log(parts['rm_negative'].clamp_min(1e-12))
        )).cpu())
    return torch.cat(values).numpy()

training_effect_rows = []
control_models = {
    'FastDepth trained on NYU': model,
    'ImageNet encoder, no depth training': FastDepthMobileNetV2(True).to(DEVICE).eval(),
    'Fully random encoder': FastDepthMobileNetV2(False).to(DEVICE).eval(),
}
for control_name, control_model in control_models.items():
    control_thresholds = calibrate_middle_magnitude(control_model, validation_loader)
    control_id = collect_middle_magnitude(control_model, test_loader, control_thresholds)
    control_ood = collect_middle_magnitude(control_model, kitti_ood_loader, control_thresholds)
    training_effect_rows.append({
        'model': control_name, **bootstrap_metrics(
            control_id, control_ood, repetitions=300
        ),
    })
    print(control_name, ood_metrics(control_id, control_ood))
training_effect_frame = pd.DataFrame(training_effect_rows).set_index('model')
display((training_effect_frame * 100).round(2))
training_effect_frame.to_csv(results_directory / 'cores_training_effect_control.csv')
del control_models
torch.cuda.empty_cache()
print('Trained/pretrained/random CORES control passed.')

In [ ]:
trained_middle_thresholds = (
    cores_thresholds['encoder_middle']['tau_positive'],
    cores_thresholds['encoder_middle']['tau_negative'],
)
clean_middle_magnitude = id_ablation_scores[('encoder_middle', 'magnitude_only')]
def gaussian_noise(images):
    return (images + torch.randn_like(images) * 0.10).clamp(0, 1)
def brightness_shift(images):
    return (images * 0.50).clamp(0, 1)
def average_blur(images):
    return F.avg_pool2d(images, kernel_size=7, stride=1, padding=3)

torch.manual_seed(CFG.seed)
corruption_rows = []
for corruption_name, transform in {
    'gaussian_noise_sigma_0.10': gaussian_noise,
    'brightness_x0.50': brightness_shift,
    'average_blur_7x7': average_blur,
}.items():
    corrupted_scores = collect_middle_magnitude(
        model, test_loader, trained_middle_thresholds, transform
    )
    corruption_rows.append({
        'corruption': corruption_name,
        **bootstrap_metrics(clean_middle_magnitude, corrupted_scores, repetitions=300),
    })
corruption_frame = pd.DataFrame(corruption_rows).set_index('corruption')
display((corruption_frame * 100).round(2))
corruption_frame.to_csv(results_directory / 'cores_nyu_corruption_detection.csv')
assert np.isfinite(corruption_frame.to_numpy()).all()
print('NYU corruption OOD controls passed.')

In [ ]:
@torch.no_grad()
def collect_middle_extrema(target_model, loader, transform=None, max_batches=32):
    maxima, minima = [], []
    for batch_index, batch in enumerate(loader):
        if batch_index >= max_batches:
            break
        images = batch['image'].to(DEVICE, non_blocking=True)
        if transform is not None:
            images = transform(images)
        response = capture_middle_response(target_model, images)
        maxima.append(response.amax(dim=(-2, -1)).cpu())
        minima.append(response.amin(dim=(-2, -1)).cpu())
    return torch.cat(maxima), torch.cat(minima)

def magnitude_score_from_extrema(maxima, minima, tau_positive, tau_negative):
    rm_positive = F.relu(maxima - tau_positive).mean(dim=1)
    rm_negative = F.relu(tau_negative - minima).mean(dim=1)
    return (10 * (
        torch.log(rm_positive.clamp_min(1e-12))
        + torch.log(rm_negative.clamp_min(1e-12))
    )).numpy()

def pure_gaussian(images):
    return (0.5 + torch.randn_like(images) * 0.25).clamp(0, 1)
def pure_uniform(images):
    return torch.rand_like(images)

torch.manual_seed(CFG.seed)
calibration_maxima, calibration_minima = collect_middle_extrema(model, validation_loader)
gaussian_maxima, gaussian_minima = collect_middle_extrema(
    model, validation_loader, pure_gaussian
)
uniform_maxima, uniform_minima = collect_middle_extrema(
    model, validation_loader, pure_uniform
)
noise_maxima = torch.cat([gaussian_maxima, uniform_maxima])
noise_minima = torch.cat([gaussian_minima, uniform_minima])
quantiles = (0.30, 0.40, 0.50, 0.60, 0.70)
calibration_grid = []
for positive_quantile in quantiles:
    tau_positive = calibration_maxima.flatten().quantile(positive_quantile).item()
    for negative_quantile in quantiles:
        tau_negative = calibration_minima.flatten().quantile(negative_quantile).item()
        calibration_id_score = magnitude_score_from_extrema(
            calibration_maxima, calibration_minima, tau_positive, tau_negative
        )
        calibration_noise_score = magnitude_score_from_extrema(
            noise_maxima, noise_minima, tau_positive, tau_negative
        )
        metrics = ood_metrics(calibration_id_score, calibration_noise_score)
        calibration_grid.append({
            'positive_quantile': positive_quantile,
            'negative_quantile': negative_quantile,
            'tau_positive': tau_positive, 'tau_negative': tau_negative, **metrics,
        })
calibration_grid_frame = pd.DataFrame(calibration_grid).sort_values(
    ['fpr95', 'auroc'], ascending=[True, False]
)
best_synthetic = calibration_grid_frame.iloc[0]
synthetic_thresholds = (best_synthetic.tau_positive, best_synthetic.tau_negative)
synthetic_id = collect_middle_magnitude(model, test_loader, synthetic_thresholds)
synthetic_ood = collect_middle_magnitude(model, kitti_ood_loader, synthetic_thresholds)
threshold_comparison = pd.DataFrame([
    {'method': 'NYU validation median',
     'tau_positive': trained_middle_thresholds[0],
     'tau_negative': trained_middle_thresholds[1],
     **ood_metrics(clean_middle_magnitude,
                   ood_ablation_scores[('encoder_middle', 'magnitude_only')])},
    {'method': 'Gaussian/uniform tuned',
     'tau_positive': synthetic_thresholds[0], 'tau_negative': synthetic_thresholds[1],
     **ood_metrics(synthetic_id, synthetic_ood)},
]).set_index('method')
display(threshold_comparison.round(5))
calibration_grid_frame.to_csv(results_directory / 'cores_synthetic_threshold_grid.csv', index=False)
threshold_comparison.to_csv(results_directory / 'cores_threshold_calibration_comparison.csv')
print('Paper-style Gaussian/uniform threshold calibration passed.')

In [ ]:
fig, axes = plt.subplots(1, len(response_modules) + 1, figsize=(20, 4))
plot_names = list(response_modules) + ['multi_layer_mean']
for axis, name in zip(axes, plot_names):
    if name == 'multi_layer_mean':
        id_values, ood_values = multi_id, multi_ood
    else:
        id_values, ood_values = normalized_id[name], normalized_ood[name]
    axis.hist(id_values, bins=35, alpha=0.65, density=True, label='NYU ID')
    axis.hist(ood_values, bins=35, alpha=0.65, density=True, label='KITTI OOD')
    axis.set_title(name); axis.set_xlabel('Normalized log CORES (higher = ID)')
    axis.grid(alpha=0.2)
axes[0].legend(); plt.tight_layout(); plt.show()

## 12. Next implementation milestone

Validate the all-kernel CORES-MDE baseline, then implement the sample-relevant-kernel adaptation and compare layer-wise versus multi-layer scores.